In [1]:
!pip install smolagents python-dotenv sqlalchemy --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.4/148.4 kB 3.5 MB/s eta 0:00:00


In [113]:
from google.colab import userdata

my_token = userdata.get('HF_TOKEN')
with open('.env', 'w') as f:
    f.write(f"HF_TOKEN={my_token}")

In [114]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [7]:
from sqlalchemy import create_engine, inspect, text

db_path = "sqlite:///shakespeare.sqlite"

if not os.path.exists("shakespeare.sqlite"):
    print("WARNING: db not found.")

engine = create_engine(db_path)

In [8]:
inspector = inspect(engine)
table_names = inspector.get_table_names()

schema = "Database Schema:\n"

for table in table_names:
    schema += f"Table: {table}\n"
    columns = inspector.get_columns(table)
    for col in columns:
        schema += f"  - {col['name']} ({col['type']})\n"

print(schema)

Database Schema:
Table: chapters
  - id (INTEGER)
  - Act (INTEGER)
  - Scene (INTEGER)
  - Description (TEXT)
  - work_id (INTEGER)
Table: characters
  - id (INTEGER)
  - CharName (TEXT)
  - Abbrev (TEXT)
  - Description (TEXT)
Table: paragraphs
  - id (INTEGER)
  - ParagraphNum (INTEGER)
  - PlainText (TEXT)
  - character_id (INTEGER)
  - chapter_id (INTEGER)
Table: works
  - id (INTEGER)
  - Title (TEXT)
  - LongTitle (TEXT)
  - Date (INTEGER)
  - GenreType (TEXT)



In [9]:
from smolagents import tool
@tool
def sql_engine(query: str) -> str:
    """
    Allows you to perform SQL queries on the table. Returns a string representation of the result.

    Args:
        query: The query to perform.
    """
    output = ""
    with engine.connect() as con:
        rows = con.execute(text(query))
        for row in rows:
            output += "\n" + str(row)
    return output

In [78]:
from smolagents import CodeAgent, InferenceClientModel

agent = CodeAgent(
    tools=[sql_engine],
    model=InferenceClientModel(model_id="Qwen/Qwen3-8B"),
)

In [104]:
question = "Give the title of the work that contains the character 'Shylock'."
evidence = "character 'Shylock' refers to CharName = 'Shylock'"

In [109]:
USER_PROMPT = f"""
Database Schema:
{schema}
Question:
{evidence}. {question}

INSTRUCTIONS:
- For each query, you need to use the 'sql_engine' tool to analyze the output of the query.
- Do not execute separate queries to fetch values and then inject them into a subsequent query. If you need to filter by a dynamic value, you have to use a SQL subquery.
- If you make exploratory queries with the 'sql_engine' tool, use LIMIT 3 in order to ensure that the tool's output does not overwhelm the agent's context window.
- Your SQL query must yield the answer on its own. You can not manipulate the data with Python.
"""

#The final answer, that you return using the 'final_answer' tool, has to contain the final SQL query, not the answer to the question.

#Eventually, provide ONLY the final SQL query enclosed within the tags <answer> and </answer>, not the answer to the query.
#Once the final, correct SQL query is constructed, execute the final step by calling final_answer(your_query_string) within the <code> block.




In [110]:
agent.run(USER_PROMPT)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Database Schema:                                                                                                │
│ Database Schema:                                                                                                │
│ Table: chapters                                                                                                 │
│   - id (INTEGER)                                                                                                │
│   - Act (INTEGER)                                                                                               │
│   - Scene (INTEGER)                                                                                             │
│   - Description (TEXT)                                                                                          │
│   - work_id (INTEGER)                                                                                           │
│ Table: characters                                                                                               │
│   - id (INTEGER)                                                                                                │
│   - CharName (TEXT)                                                                                             │
│   - Abbrev (TEXT)                                                                                               │
│   - Description (TEXT)                                                                                          │
│ Table: paragraphs                                                                                               │
│   - id (INTEGER)                                                                                                │
│   - ParagraphNum (INTEGER)                                                                                      │
│   - PlainText (TEXT)                                                                                            │
│   - character_id (INTEGER)                                                                                      │
│   - chapter_id (INTEGER)                                                                                        │
│ Table: works                                                                                                    │
│   - id (INTEGER)                                                                                                │
│   - Title (TEXT)                                                                                                │
│   - LongTitle (TEXT)                                                                                            │
│   - Date (INTEGER)                                                                                              │
│   - GenreType (TEXT)                                                                                            │
│                                                                                                                 │
│ Question:                                                                                                       │
│ character 'Shylock' refers to CharName = 'Shylock'. Give the title of the work that contains the character      │
│ 'Shylock'.                                                                                                      │
│                                                                                                                 │
│ INSTRUCTIONS:                                                                                                   │
│ - For each query, you need to use the 'sql_engine' tool to analyze the output of the query.                     │
│ - Do not execute separate queries to fetch values and 

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  query = """                                                                                                      
  SELECT works.LongTitle                                                                                           
  FROM characters                                                                                                  
  JOIN works ON characters.work_id = works.id                                                                      
  WHERE characters.CharName = 'Shylock';                                                                           
  """                                                                                                              
  result = sql_engine(query=query)                                                                                 
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'result = sql_engine(query=query)' due to: OperationalError: 
(sqlite3.OperationalError) no such column: characters.work_id
[SQL: 
SELECT works.LongTitle
FROM characters
JOIN works ON characters.work_id = works.id
WHERE characters.CharName = 'Shylock';
\]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[Step 1: Duration 11.32 seconds| Input tokens: 2,366 | Output tokens: 691]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The Merchant of Venice")                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The Merchant of Venice

[Step 3: Duration 4.87 seconds| Input tokens: 8,021 | Output tokens: 1,504]

'The Merchant of Venice'

In [ ]:
print(agent.memory.steps[1].model_input_messages[0].content[0]['text'])

You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.

At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '<code>', and closed with '</code>'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.

Here are a few examples 